# Arousal Prediction v10
**One focused change:** Apply confirmed optimal thresholds [1.2, 2.1, 3.1, 4.1] instead of default [1.5, 2.5, 3.5, 4.5]
- Model consistently under-predicts arousal (confirmed across all LOPO folds)
- Shifting boundaries left corrects this bias
- Everything else identical to v4

In [1]:
!pip install -q pandas numpy scipy scikit-learn lightgbm imbalanced-learn


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.signal import welch
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
from imblearn.over_sampling import ADASYN, SMOTE
np.random.seed(42)
print('Libraries loaded.')

Libraries loaded.


## 1. Load Data

In [3]:
DATA = '.'
train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')
trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainbrain = pd.read_csv(f'{DATA}/train-brain.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')
testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testbrain  = pd.read_csv(f'{DATA}/test-brain.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')
print(f'Train: {len(train_labels)} | Test: {len(test_labels)}')
print(train_labels['arousal'].value_counts().sort_index())

Train: 1456 | Test: 1496
arousal
1     55
2    430
3    554
4    345
5     72
Name: count, dtype: int64


## 2. Feature Helpers

In [4]:
WINDOW = 5000
BVP_WINDOW = 10000
BRAIN_WINDOW = 10000
IBI_WINDOW = 20000
BASELINE_W = 30000

def assign_windows_forward(sensor_df, labels_df, window):
    results = []
    for pid in labels_df['pid'].unique():
        lbl = labels_df[labels_df['pid']==pid].sort_values('timestamp')
        sen = sensor_df[sensor_df['pid']==pid].copy()
        if len(sen) == 0: continue
        lbl_ts = lbl['timestamp'].values
        bins = np.append(lbl_ts, lbl_ts[-1] + window)
        sen['label_ts'] = pd.cut(sen['timestamp'], bins=bins,
                                   labels=lbl_ts, right=False, include_lowest=True)
        results.append(sen.dropna(subset=['label_ts']))
    if not results: return pd.DataFrame()
    out = pd.concat(results, ignore_index=True)
    out['label_ts'] = out['label_ts'].astype(np.int64)
    return out

def assign_windows_lookback(sensor_df, labels_df, window):
    results = []
    for pid in labels_df['pid'].unique():
        lbl = labels_df[labels_df['pid']==pid].sort_values('timestamp')
        sen = sensor_df[sensor_df['pid']==pid].copy()
        if len(sen) == 0: continue
        for ts in lbl['timestamp'].values:
            chunk = sen[(sen['timestamp'] >= ts-window) & (sen['timestamp'] < ts)]
            if len(chunk) > 0:
                chunk = chunk.copy()
                chunk['label_ts'] = ts
                chunk['pid'] = pid
                results.append(chunk)
    if not results: return pd.DataFrame()
    out = pd.concat(results, ignore_index=True)
    out['label_ts'] = out['label_ts'].astype(np.int64)
    return out

def safe_stats(vals, prefix):
    a = np.array(vals, dtype=float); a = a[~np.isnan(a)]
    if len(a) == 0:
        return {f'{prefix}_{k}': np.nan for k in ['mean','std','min','max','range','q25','q75','iqr','skew','kurt','rms','count']}
    return {
        f'{prefix}_mean': float(np.mean(a)),
        f'{prefix}_std':  float(np.std(a)) if len(a)>1 else 0.0,
        f'{prefix}_min':  float(np.min(a)),
        f'{prefix}_max':  float(np.max(a)),
        f'{prefix}_range':float(np.ptp(a)),
        f'{prefix}_q25':  float(np.percentile(a,25)),
        f'{prefix}_q75':  float(np.percentile(a,75)),
        f'{prefix}_iqr':  float(np.percentile(a,75)-np.percentile(a,25)),
        f'{prefix}_skew': float(stats.skew(a)) if len(a)>2 else 0.0,
        f'{prefix}_kurt': float(stats.kurtosis(a)) if len(a)>2 else 0.0,
        f'{prefix}_rms':  float(np.sqrt(np.mean(a**2))),
        f'{prefix}_count':float(len(a)),
    }

def spectral_features(vals, fs, prefix):
    a = np.array(vals, dtype=float); a = a[~np.isnan(a)]
    if len(a) < 8:
        return {f'{prefix}_lf': np.nan, f'{prefix}_hf': np.nan, f'{prefix}_lf_hf': np.nan}
    try:
        f, psd = welch(a, fs=fs, nperseg=min(len(a), 64))
        lf = np.trapz(psd[(f>=0.04)&(f<=0.15)], f[(f>=0.04)&(f<=0.15)])
        hf = np.trapz(psd[(f>=0.15)&(f<=0.40)], f[(f>=0.15)&(f<=0.40)])
        return {f'{prefix}_lf': float(lf), f'{prefix}_hf': float(hf), f'{prefix}_lf_hf': float(lf/(hf+1e-9))}
    except:
        return {f'{prefix}_lf': np.nan, f'{prefix}_hf': np.nan, f'{prefix}_lf_hf': np.nan}

print('Helpers defined.')

Helpers defined.


## 3. Feature Extraction

In [5]:
def build_features(labels_df, bvp_df, eda_df, temp_df, hr_df, ibi_df, brain_df, acc_df):
    print(' Forward windows...')
    bvp_w   = assign_windows_forward(bvp_df,   labels_df, BVP_WINDOW)
    eda_w   = assign_windows_forward(eda_df,   labels_df, WINDOW)
    temp_w  = assign_windows_forward(temp_df,  labels_df, WINDOW)
    hr_w    = assign_windows_forward(hr_df,    labels_df, WINDOW)
    ibi_w   = assign_windows_forward(ibi_df,   labels_df, IBI_WINDOW)
    brain_w = assign_windows_forward(brain_df, labels_df, BRAIN_WINDOW)
    acc_w   = assign_windows_forward(acc_df,   labels_df, WINDOW)
    print(' Lookback baseline...')
    eda_b  = assign_windows_lookback(eda_df,  labels_df, BASELINE_W)
    temp_b = assign_windows_lookback(temp_df, labels_df, BASELINE_W)
    hr_b   = assign_windows_lookback(hr_df,   labels_df, BASELINE_W)
    acc_b  = assign_windows_lookback(acc_df,  labels_df, BASELINE_W)
    print(' BVP...')
    bvp_agg = bvp_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    bvp_feats = []
    for _, r in bvp_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'bvp'))
        d.update(spectral_features(r['value'], fs=64, prefix='bvp_spec'))
        a = np.array(r['value'])
        d['bvp_zcr'] = float(np.sum(np.diff(np.sign(np.diff(a)))!=0)/len(a)) if len(a)>1 else 0
        bvp_feats.append(d)
    bvp_feat_df = pd.DataFrame(bvp_feats)
    print(' EDA...')
    eda_agg = eda_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    eda_b_mean = eda_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    eda_b_mean.columns = ['pid','label_ts','eda_baseline']
    eda_feats = []
    for _, r in eda_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'eda'))
        a = np.array(r['value'], dtype=float)
        if len(a) > 2:
            d['eda_slope'] = float(np.polyfit(np.arange(len(a)), a, 1)[0])
            d['eda_peaks'] = float(np.sum((np.diff(np.sign(np.diff(a))))<-1.9))
        else:
            d['eda_slope'] = d['eda_peaks'] = np.nan
        eda_feats.append(d)
    eda_feat_df = pd.DataFrame(eda_feats)
    eda_feat_df = eda_feat_df.merge(eda_b_mean, on=['pid','label_ts'], how='left')
    eda_feat_df['eda_delta'] = eda_feat_df['eda_mean'] - eda_feat_df['eda_baseline']
    print(' TEMP...')
    temp_feat_df = temp_w.groupby(['pid','label_ts'])['value'].agg(
        temp_mean='mean', temp_std='std', temp_min='min',
        temp_max='max', temp_range=lambda x: x.max()-x.min()).reset_index()
    temp_b_mean = temp_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    temp_b_mean.columns = ['pid','label_ts','temp_baseline']
    temp_feat_df = temp_feat_df.merge(temp_b_mean, on=['pid','label_ts'], how='left')
    temp_feat_df['temp_delta'] = temp_feat_df['temp_mean'] - temp_feat_df['temp_baseline']
    print(' HR...')
    hr_feat_df = hr_w.groupby(['pid','label_ts'])['value'].agg(
        hr_mean='mean', hr_std='std', hr_min='min',
        hr_max='max', hr_range=lambda x: x.max()-x.min()).reset_index()
    hr_b_mean = hr_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    hr_b_mean.columns = ['pid','label_ts','hr_baseline']
    hr_feat_df = hr_feat_df.merge(hr_b_mean, on=['pid','label_ts'], how='left')
    hr_feat_df['hr_delta'] = hr_feat_df['hr_mean'] - hr_feat_df['hr_baseline']
    print(' IBI/HRV...')
    ibi_agg = ibi_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    ibi_feats = []
    for _, r in ibi_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'ibi'))
        a = np.array(r['value'], dtype=float); a = a[~np.isnan(a)]
        if len(a) > 1:
            diffs = np.diff(a)
            d['ibi_rmssd'] = float(np.sqrt(np.mean(diffs**2)))
            d['ibi_sdnn']  = float(np.std(a))
            d['ibi_pnn50'] = float(np.mean(np.abs(diffs)>50))
        else:
            d['ibi_rmssd'] = d['ibi_sdnn'] = d['ibi_pnn50'] = np.nan
        d.update(spectral_features(r['value'], fs=4, prefix='ibi_spec'))
        ibi_feats.append(d)
    ibi_feat_df = pd.DataFrame(ibi_feats)
    print(' Brain...')
    brain_cols = ['delta','lowAlpha','highAlpha','lowBeta','highBeta','lowGamma','middleGamma','theta']
    brain_agg = brain_w.groupby(['pid','label_ts'])[brain_cols].agg(['mean','std'])
    brain_agg.columns = [f'brain_{c}_{s}' for c,s in brain_agg.columns]
    brain_agg = brain_agg.reset_index()
    brain_agg['brain_alpha_beta']  = ((brain_agg['brain_lowAlpha_mean']+brain_agg['brain_highAlpha_mean']) /
                                       (brain_agg['brain_lowBeta_mean']+brain_agg['brain_highBeta_mean']+1e-9))
    brain_agg['brain_theta_alpha'] = (brain_agg['brain_theta_mean'] /
                                       (brain_agg['brain_lowAlpha_mean']+brain_agg['brain_highAlpha_mean']+1e-9))
    brain_agg['brain_engagement']  = (brain_agg['brain_lowBeta_mean'] /
                                       (brain_agg['brain_theta_mean']+brain_agg['brain_lowAlpha_mean']+1e-9))
    print(' ACC...')
    acc_w['mag'] = np.sqrt(acc_w['x']**2 + acc_w['y']**2 + acc_w['z']**2)
    acc_b['mag'] = np.sqrt(acc_b['x']**2 + acc_b['y']**2 + acc_b['z']**2)
    acc_agg = acc_w.groupby(['pid','label_ts'])['mag'].apply(list).reset_index()
    acc_b_mean = acc_b.groupby(['pid','label_ts'])['mag'].mean().reset_index()
    acc_b_mean.columns = ['pid','label_ts','acc_baseline']
    acc_feats = []
    for _, r in acc_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['mag'], 'acc'))
        a = np.array(r['mag'], dtype=float)
        if len(a) > 2:
            jerk = np.diff(a)
            d['acc_jerk_mean'] = float(np.mean(np.abs(jerk)))
            d['acc_jerk_std']  = float(np.std(jerk))
        else:
            d['acc_jerk_mean'] = d['acc_jerk_std'] = np.nan
        acc_feats.append(d)
    acc_feat_df = pd.DataFrame(acc_feats)
    acc_feat_df = acc_feat_df.merge(acc_b_mean, on=['pid','label_ts'], how='left')
    acc_feat_df['acc_delta'] = acc_feat_df['acc_mean'] - acc_feat_df['acc_baseline']
    print(' Merging...')
    base = labels_df[['id','pid','timestamp']].copy()
    base['label_ts'] = base['timestamp']
    merged = base
    for fdf in [bvp_feat_df, eda_feat_df, temp_feat_df, hr_feat_df, ibi_feat_df, brain_agg, acc_feat_df]:
        fdf['label_ts'] = fdf['label_ts'].astype(np.int64)
        merged = merged.merge(fdf, on=['pid','label_ts'], how='left')
    return merged.drop(columns=['label_ts'])

print('Extracting TRAIN features...')
train_feats = build_features(train_labels, trainbvp, traineda, traintemp, trainhr, trainibi, trainbrain, trainacc)
print(f'Train: {train_feats.shape}, missing: {train_feats.isnull().mean().mean():.2%}')

Extracting TRAIN features...
 Forward windows...
 Lookback baseline...
 BVP...
 EDA...
 TEMP...
 HR...
 IBI/HRV...
 Brain...
 ACC...
 Merging...
Train: (1456, 102), missing: 19.11%


In [6]:
print('Extracting TEST features...')
test_feats = build_features(test_labels, testbvp, testeda, testtemp, testhr, testibi, testbrain, testacc)
print(f'Test: {test_feats.shape}, missing: {test_feats.isnull().mean().mean():.2%}')

Extracting TEST features...
 Forward windows...
 Lookback baseline...
 BVP...
 EDA...
 TEMP...
 HR...
 IBI/HRV...
 Brain...
 ACC...
 Merging...
Test: (1496, 102), missing: 19.73%


## 4. Lags + Prep + Outliers + Normalisation

In [7]:
LAG_COLS = ['eda_mean','eda_min','eda_q75','eda_slope','eda_delta',
            'temp_mean','temp_min','temp_delta','hr_mean','hr_std','hr_delta',
            'bvp_mean','bvp_skew','acc_mean','acc_max','acc_delta','ibi_rmssd','ibi_sdnn']

def add_lag_features(feat_df, lag_cols, lags=[1,2]):
    parts = []
    for pid in feat_df['pid'].unique():
        sub = feat_df[feat_df['pid']==pid].sort_values('timestamp').copy()
        for lag in lags:
            for col in lag_cols:
                if col in sub.columns:
                    sub[f'{col}_lag{lag}'] = sub[col].shift(lag)
        parts.append(sub)
    return pd.concat(parts).sort_values('id').reset_index(drop=True)

print('Adding lag features...')
train_w_lags = add_lag_features(train_feats, LAG_COLS)
test_w_lags  = add_lag_features(test_feats,  LAG_COLS)

train_df = train_w_lags.merge(train_labels[['id','arousal']], on='id', how='left')
test_df  = test_w_lags.copy()
feat_cols = [c for c in train_df.columns if c not in ['id','pid','timestamp','arousal']]

sensor_groups = {'bvp': [c for c in feat_cols if c.startswith('bvp')],
                 'eda': [c for c in feat_cols if c.startswith('eda')],
                 'temp':[c for c in feat_cols if c.startswith('temp')],
                 'hr':  [c for c in feat_cols if c.startswith('hr')],
                 'ibi': [c for c in feat_cols if c.startswith('ibi')],
                 'brain':[c for c in feat_cols if c.startswith('brain')],
                 'acc': [c for c in feat_cols if c.startswith('acc')]}
for grp, cols in sensor_groups.items():
    if cols:
        train_df[f'{grp}_missing'] = train_df[cols].isnull().any(axis=1).astype(int)
        test_df[f'{grp}_missing']  = test_df[cols].isnull().any(axis=1).astype(int)
feat_cols = [c for c in train_df.columns if c not in ['id','pid','timestamp','arousal']]
indicator_cols = [f'{g}_missing' for g in sensor_groups]

def impute_by_pid(df, cols):
    df = df.copy()
    df[cols] = df[cols].fillna(df.groupby('pid')[cols].transform('median'))
    df[cols] = df[cols].fillna(df[cols].median())
    return df

train_df = impute_by_pid(train_df, feat_cols)
test_df  = impute_by_pid(test_df,  feat_cols)
test_df[feat_cols] = test_df[feat_cols].fillna(train_df[feat_cols].median())

X_all = train_df[feat_cols].values
y_all = train_df['arousal'].values
outlier_mask = np.zeros(len(train_df), dtype=bool)
for cls in np.unique(y_all):
    idx = np.where(y_all == cls)[0]
    contamination = 0.05 if len(idx) < 100 else 0.07
    iso = IsolationForest(n_estimators=100, contamination=contamination, random_state=42)
    preds = iso.fit_predict(X_all[idx])
    n_out = np.sum(preds==-1)
    print(f' Class {cls} ({len(idx)}): {n_out} outliers')
    outlier_mask[idx[preds==-1]] = True
train_clean = train_df[~outlier_mask].copy().reset_index(drop=True)
print(f'Kept: {len(train_clean)} / {len(train_df)}')

norm_cols = [c for c in feat_cols if not c.endswith('_missing')]
def zscore_by_pid(df, cols):
    df = df.copy()
    for pid in df['pid'].unique():
        mask = df['pid'] == pid
        sub = df.loc[mask, cols]
        mu = sub.mean(); sig = sub.std().replace(0, np.nan)
        df.loc[mask, cols] = (sub - mu) / sig
    df[cols] = df[cols].fillna(0)
    return df

train_norm = zscore_by_pid(train_clean, norm_cols)
test_norm  = zscore_by_pid(test_df,     norm_cols)
print('Done.')

Adding lag features...
 Class 1 (55): 3 outliers
 Class 2 (430): 31 outliers
 Class 3 (554): 39 outliers
 Class 4 (345): 25 outliers
 Class 5 (72): 4 outliers
Kept: 1354 / 1456
Done.


## 5. Feature Selection

In [8]:
X_sel = train_norm[feat_cols].values
y_sel = train_norm['arousal'].values.astype(float)
rf_sel = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_sel.fit(X_sel, y_sel)
imp_df = pd.DataFrame({'feature': feat_cols, 'importance': rf_sel.feature_importances_}).sort_values('importance', ascending=False)
top_features = imp_df.head(60)['feature'].tolist()
for ind in indicator_cols:
    if ind not in top_features and ind in feat_cols:
        top_features.append(ind)
print(f'Selected {len(top_features)} features')
print(imp_df.head(10)[['feature','importance']].to_string(index=False))

Selected 67 features
      feature  importance
    ibi_count    0.070726
    acc_count    0.070223
    bvp_count    0.065115
    eda_count    0.060736
      eda_min    0.034481
  hr_baseline    0.033559
     ibi_skew    0.030174
temp_baseline    0.025218
 eda_min_lag1    0.023826
hr_delta_lag2    0.017237


## 6. Oversampling + Class Weights

In [9]:
X_train = train_norm[top_features].values
y_train = train_norm['arousal'].values
unique, counts = np.unique(y_train, return_counts=True)
target = {cls: max(cnt, 150) for cls, cnt in zip(unique.astype(int), counts)}
try:
    res = ADASYN(sampling_strategy=target, n_neighbors=min(4, min(counts)-1), random_state=42)
    X_res, y_res = res.fit_resample(X_train, y_train)
    print('ADASYN succeeded.')
except:
    res = SMOTE(sampling_strategy=target, k_neighbors=min(3, min(counts)-1), random_state=42)
    X_res, y_res = res.fit_resample(X_train, y_train)
class_counts_res = dict(zip(*np.unique(y_res, return_counts=True)))
total_res = len(y_res)
class_weights = {cls: total_res/(len(class_counts_res)*cnt) for cls,cnt in class_counts_res.items()}
class_weights[1] = class_weights[1] * 2.0
class_weights[5] = class_weights[5] * 2.0
sample_weights = np.array([class_weights[y] for y in y_res])
print('Class weights:', {k: round(v,3) for k,v in sorted(class_weights.items())})

ADASYN succeeded.
Class weights: {np.int64(1): np.float64(3.926), np.int64(2): np.float64(0.767), np.int64(3): np.float64(0.595), np.int64(4): np.float64(0.957), np.int64(5): np.float64(4.343)}


## 7. LOPO CV

In [10]:
def apply_thresholds(preds, t):
    out = np.ones(len(preds), dtype=int)
    out[preds >= t[0]] = 2
    out[preds >= t[1]] = 3
    out[preds >= t[2]] = 4
    out[preds >= t[3]] = 5
    return out

def ordinal_clip(pred):
    return np.clip(np.round(pred), 1, 5).astype(int)

rf_params  = dict(n_estimators=500, max_depth=10, min_samples_leaf=4,
                  min_samples_split=8, max_features='sqrt', max_samples=0.8,
                  random_state=42, n_jobs=-1)
lgb_params = dict(objective='regression_l1', metric='mae', n_estimators=400,
                  learning_rate=0.04, num_leaves=31, max_depth=5, min_child_samples=20,
                  subsample=0.75, colsample_bytree=0.75, reg_alpha=0.5, reg_lambda=1.0,
                  random_state=42, verbose=-1)

# v10 confirmed optimal thresholds (from v9 LOPO analysis)
OPTIMAL_THRESH = (1.2, 2.1, 3.1, 4.1)
DEFAULT_THRESH  = (1.5, 2.5, 3.5, 4.5)

pids = train_norm['pid'].unique()
lopo_maes = {'rf':[], 'lgb':[], 'ens':[]}
lopo_acc  = {'default':[], 'optimal':[]}
lopo_pid_results = []
print(f'LOPO CV ({len(pids)} folds)...\n')

for held_pid in pids:
    mask_val = train_norm['pid'] == held_pid
    mask_tr  = ~mask_val
    X_tr  = train_norm.loc[mask_tr,  top_features].values
    y_tr  = train_norm.loc[mask_tr,  'arousal'].values.astype(float)
    X_val = train_norm.loc[mask_val, top_features].values
    y_val = train_norm.loc[mask_val, 'arousal'].values

    u_tr, c_tr = np.unique(y_tr, return_counts=True)
    fold_target = {cls: max(cnt, 100) for cls,cnt in zip(u_tr.astype(int), c_tr)}
    safe_k = max(1, min(3, min(c_tr)-1))
    try:
        sm = ADASYN(sampling_strategy=fold_target, n_neighbors=safe_k, random_state=42)
        X_tr_s, y_tr_s = sm.fit_resample(X_tr, y_tr)
    except:
        sm = SMOTE(sampling_strategy=fold_target, k_neighbors=safe_k, random_state=42)
        X_tr_s, y_tr_s = sm.fit_resample(X_tr, y_tr)
    sw = np.array([class_weights.get(int(y), 1.0) for y in y_tr_s])

    m_rf  = RandomForestRegressor(**rf_params)
    m_rf.fit(X_tr_s, y_tr_s, sample_weight=sw)
    m_lgb = lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(X_tr_s, y_tr_s, sample_weight=sw)

    p_rf_raw  = m_rf.predict(X_val)
    p_lgb_raw = m_lgb.predict(X_val)

    best_w, best_mae = 0.6, float('inf')
    for w in [0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
        p = ordinal_clip(w*p_rf_raw + (1-w)*p_lgb_raw)
        m = mean_absolute_error(y_val, p)
        if m < best_mae: best_mae, best_w = m, w

    raw_ens = best_w*p_rf_raw + (1-best_w)*p_lgb_raw

    # Calibrate
    tq = np.percentile(y_tr, np.linspace(0,100,1000))
    pq = np.percentile(raw_ens, np.linspace(0,100,1000))
    cf = interp1d(pq, tq, bounds_error=False, fill_value=(tq[0], tq[-1]))
    calib = cf(raw_ens)

    acc_def = np.mean(apply_thresholds(calib, DEFAULT_THRESH)  == y_val)
    acc_opt = np.mean(apply_thresholds(calib, OPTIMAL_THRESH)  == y_val)
    mae_ens = best_mae

    lopo_maes['rf'].append(mean_absolute_error(y_val, ordinal_clip(p_rf_raw)))
    lopo_maes['lgb'].append(mean_absolute_error(y_val, ordinal_clip(p_lgb_raw)))
    lopo_maes['ens'].append(mae_ens)
    lopo_acc['default'].append(acc_def)
    lopo_acc['optimal'].append(acc_opt)
    lopo_pid_results.append({'pid': held_pid, 'ens_mae': mae_ens,
                              'acc_def': acc_def, 'acc_opt': acc_opt, 'best_w': best_w})
    print(f'  [{held_pid}] MAE={mae_ens:.3f} acc_default={acc_def:.3f} acc_optimal={acc_opt:.3f} (w_RF={best_w})')

print()
print(f'ENS  LOPO MAE: {np.mean(lopo_maes["ens"]):.4f} ± {np.std(lopo_maes["ens"]):.4f}')
print(f'Accuracy default thresh: {np.mean(lopo_acc["default"]):.4f}')
print(f'Accuracy optimal thresh: {np.mean(lopo_acc["optimal"]):.4f}')
print(f'Gain from thresholds:    {(np.mean(lopo_acc["optimal"])-np.mean(lopo_acc["default"]))*100:+.2f}%')

LOPO CV (11 folds)...

  [01Z2] MAE=0.900 acc_default=0.275 acc_optimal=0.275 (w_RF=1.0)
  [70N8] MAE=0.453 acc_default=0.242 acc_optimal=0.242 (w_RF=0.8)
  [7PF3] MAE=0.786 acc_default=0.255 acc_optimal=0.255 (w_RF=0.5)
  [CQ2G] MAE=1.066 acc_default=0.132 acc_optimal=0.140 (w_RF=0.8)
  [D1XP] MAE=0.200 acc_default=0.300 acc_optimal=0.300 (w_RF=1.0)
  [DT5C] MAE=0.967 acc_default=0.156 acc_optimal=0.156 (w_RF=0.5)
  [F1ZM] MAE=0.817 acc_default=0.229 acc_optimal=0.229 (w_RF=1.0)
  [LIUY] MAE=1.371 acc_default=0.202 acc_optimal=0.202 (w_RF=1.0)
  [SE4Q] MAE=0.664 acc_default=0.213 acc_optimal=0.213 (w_RF=1.0)
  [TPQI] MAE=0.707 acc_default=0.315 acc_optimal=0.315 (w_RF=0.4)
  [Y21H] MAE=0.878 acc_default=0.244 acc_optimal=0.244 (w_RF=1.0)

ENS  LOPO MAE: 0.8008 ± 0.2937
Accuracy default thresh: 0.2331
Accuracy optimal thresh: 0.2338
Gain from thresholds:    +0.08%


## 8. Final Training + Prediction

In [11]:
print('Training final models...')
inv = np.array([1/np.mean(lopo_maes['rf']), 1/np.mean(lopo_maes['lgb'])])
w_rf, w_lgb = inv / inv.sum()
print(f'Final weights: RF={w_rf:.3f}, LGB={w_lgb:.3f}')

rf_final  = RandomForestRegressor(**rf_params)
rf_final.fit(X_res, y_res.astype(float), sample_weight=sample_weights)
print(' RF done.')
lgb_final = lgb.LGBMRegressor(**lgb_params)
lgb_final.fit(X_res, y_res.astype(float), sample_weight=sample_weights)
print(' LGB done.')

X_test  = test_norm[top_features].values
raw_rf  = rf_final.predict(X_test)
raw_lgb = lgb_final.predict(X_test)
raw_ens = w_rf * raw_rf + w_lgb * raw_lgb

# Rank-based calibration (same as v4)
train_y_float = train_norm['arousal'].values.astype(float)
train_q = np.percentile(train_y_float, np.linspace(0,100,1000))
pred_q  = np.percentile(raw_ens,       np.linspace(0,100,1000))
calib_fn = interp1d(pred_q, train_q, bounds_error=False, fill_value=(train_q[0], train_q[-1]))
calib_preds = calib_fn(raw_ens)

# Apply optimal thresholds [1.2, 2.1, 3.1, 4.1]
pred_classes = apply_thresholds(calib_preds, OPTIMAL_THRESH)

print('\nPrediction distribution (optimal thresholds):')
u, c = np.unique(pred_classes, return_counts=True)
for cls, cnt in zip(u, c):
    pct = cnt/len(pred_classes)*100
    train_pct = (train_labels['arousal']==cls).mean()*100
    print(f'  Class {cls}: {cnt:4d} ({pct:.1f}%) | Train {train_pct:.1f}% | Δ={pct-train_pct:+.1f}%')

Training final models...
Final weights: RF=0.545, LGB=0.455
 RF done.
 LGB done.

Prediction distribution (optimal thresholds):
  Class 1:   57 (3.8%) | Train 3.8% | Δ=+0.0%
  Class 2:  441 (29.5%) | Train 29.5% | Δ=-0.1%
  Class 3:  568 (38.0%) | Train 38.0% | Δ=-0.1%
  Class 4:  353 (23.6%) | Train 23.7% | Δ=-0.1%
  Class 5:   77 (5.1%) | Train 4.9% | Δ=+0.2%


## 9. Save Submission

In [12]:
submission = pd.DataFrame({'id': test_labels['id'].values, 'arousal': pred_classes})
assert len(submission) == 1496
assert submission['arousal'].between(1,5).all()
submission.to_csv('submission_v10.csv', index=False)

print('Saved: submission_v10.csv')
print(submission['arousal'].value_counts().sort_index())
print(f'\n--- Summary ---')
print(f'v4  LB: 0.23191 | LOPO MAE: 0.8002')
print(f'v10 LOPO MAE: {np.mean(lopo_maes["ens"]):.4f}')
print(f'v10 LOPO acc (default thresh): {np.mean(lopo_acc["default"]):.4f}')
print(f'v10 LOPO acc (optimal thresh): {np.mean(lopo_acc["optimal"]):.4f}')
print(f'Thresholds used: {OPTIMAL_THRESH}')

Saved: submission_v10.csv
arousal
1     57
2    441
3    568
4    353
5     77
Name: count, dtype: int64

--- Summary ---
v4  LB: 0.23191 | LOPO MAE: 0.8002
v10 LOPO MAE: 0.8008
v10 LOPO acc (default thresh): 0.2331
v10 LOPO acc (optimal thresh): 0.2338
Thresholds used: (1.2, 2.1, 3.1, 4.1)
